# VedaVision — Single-Leaf Preprocessing & Feature Extraction
### Step-by-step visualisation of every pipeline stage

**How to use:**
1. Set `IMAGE_PATH` and `SPECIES` in the *Configuration* cell.
2. Run all cells top-to-bottom (`Kernel → Restart & Run All`).
3. Every stage prints its intermediate output so you can inspect what changed.

> **Masking section** shows ALL 9 internal stages of the background-removal algorithm.  
> **Feature extraction section** shows each feature group with visual overlays.


## 0 · Imports & helpers

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")

# ── Make the module root importable ──────────────────────────────────────
import os
MODULE_ROOT = os.path.abspath(".")   # adjust if needed
for _ in range(3):
    if os.path.isdir(os.path.join(MODULE_ROOT, "preprocessing")):
        break
    parent = os.path.dirname(MODULE_ROOT)
    if parent == MODULE_ROOT:
        break
    MODULE_ROOT = parent
if MODULE_ROOT not in sys.path:
    sys.path.insert(0, MODULE_ROOT)

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from skimage.morphology import skeletonize

# ── Plotting helpers ──────────────────────────────────────────────────────
def _bgr2rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def _show(images, titles, cmap_list=None, figsize=None, suptitle=None):
    """Generic multi-panel display."""
    n = len(images)
    if figsize is None:
        figsize = (5 * n, 5)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for ax, img, title, cmap in zip(
        axes, images, titles,
        cmap_list or ["gray"] * n
    ):
        if img.ndim == 3:
            ax.imshow(_bgr2rgb(img))
        else:
            ax.imshow(img, cmap=cmap)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

def _overlay_mask(img_bgr, mask, colour=(0, 255, 0), alpha=0.35):
    """Return a copy of img with the mask tinted in `colour`."""
    out = _bgr2rgb(img_bgr).copy()
    tint = np.zeros_like(out)
    tint[mask > 0] = colour[::-1]   # colour is BGR → flip for RGB display
    return cv2.addWeighted(out, 1 - alpha, tint, alpha, 0)

print(f"✓ Imports OK  (MODULE_ROOT={MODULE_ROOT})")


## 1 · Configuration — set your image path here

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  EDIT THESE TWO LINES                                ║
# ╚══════════════════════════════════════════════════════╝
IMAGE_PATH = "../dataset/raw/kattakumanjal.jpg"   # ← replace with your image
SPECIES    = "Kattakumanjal"               # ← replace with species label
VIEW_SIDE  = "top"                            # "top" or "bottom"

# ── Load raw image ────────────────────────────────────────────────────────
img_orig = cv2.imread(IMAGE_PATH)
assert img_orig is not None, f"cv2.imread failed — check IMAGE_PATH: {IMAGE_PATH}"
print(f"✓ Loaded image  shape={img_orig.shape}  dtype={img_orig.dtype}")

_show([img_orig], [f"Original — {os.path.basename(IMAGE_PATH)}"],
      suptitle="Stage 0 · Raw Input Image")


## 2 · Preprocessing — Step 1: Letterbox Resize
Scales the longest side to 512 px and pads the short side with white,
preserving the true aspect ratio so shape features are not distorted.


In [ ]:
from preprocessing.shared.resize import letterbox_resize
from preprocessing.config import TARGET_LONG

img_resized, resize_meta = letterbox_resize(img_orig, TARGET_LONG)

print("Resize metadata:")
for k, v in resize_meta.items():
    print(f"  {k:12s} = {v}")

_show(
    [img_orig, img_resized],
    [f"Original  {img_orig.shape[1]}×{img_orig.shape[0]}",
     f"Letterboxed  {img_resized.shape[1]}×{img_resized.shape[0]}  (512×512)"],
    suptitle="Step 1 · Letterbox Resize"
)


## 3 · Preprocessing — Step 2: Background Removal (all 9 stages)
The masking module runs a 9-stage pipeline.  
Every intermediate mask is captured below so you can see exactly what changed at each stage.


In [ ]:
# ── Precompute shared inputs (mirrors masking.py select_mask internals) ──
from preprocessing.config import MIN_COMP_FRAC, SIGMA_THRESH
from preprocessing.shared.masking import (
    _build_seed, _learn_leaf_model, _build_candidate_map,
    _grow_seed, _select_structure, _build_rachis_mask,
    _fill_holes, _remove_noise, select_mask, qc_check,
)

img_area      = img_resized.shape[0] * img_resized.shape[1]
img_lab_u8    = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
img_lab_float = img_lab_u8.astype(np.float32)
hsv           = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
is_padding    = np.all(img_resized >= 252, axis=2)
k3            = np.ones((3, 3), np.uint8)

print("✓ Shared colour-space inputs ready")
print(f"  Image area    : {img_area} px")
print(f"  Padding pixels: {is_padding.sum()} px")


### 3.1 · Stage 1 — Seed Detection
Identifies high-confidence green-tissue pixels using ExG (Excess Green Index),
HSV saturation, and LAB lightness. Falls back to a relaxed threshold for
dark-pigmented species if Tier-1 coverage < 1 %.


In [ ]:
seed, seed_cov, seed_relaxed = _build_seed(
    img_resized, img_lab_float, hsv, is_padding, k3, MIN_COMP_FRAC, img_area
)

# Compute ExG for visualisation
img_f = img_resized.astype(np.float32)
exg = 2.0 * img_f[:,:,1] - img_f[:,:,2] - img_f[:,:,0]
exg_vis = np.clip((exg + 50) / 200 * 255, 0, 255).astype(np.uint8)

print(f"Seed coverage : {seed_cov:.2f}%")
print(f"Tier-2 used   : {seed_relaxed}  (True = dark species fallback)")
print(f"Seed pixels   : {(seed > 0).sum()}")

_show(
    [img_resized, exg_vis, seed, _overlay_mask(img_resized, seed, colour=(0, 255, 0))],
    ["Letterboxed input", "ExG map (brightness = ExG value)",
     "Stage 1 · Seed mask", "Seed overlay on image"],
    cmap_list=[None, "RdYlGn", "gray", None],
    figsize=(20, 5),
    suptitle="Stage 1 · Seed Detection (green = ExG > threshold)"
)


### 3.2 · Stage 2 — Per-Image Leaf Colour Model
Computes LAB mean ± std from seed pixels to build a species-adaptive colour gate.
Std is clamped to ≥ 8 to avoid over-tight gates on spectrally uniform leaves.


In [ ]:
mean_lab, std_lab = _learn_leaf_model(img_lab_float, seed)

print("Leaf colour model (LAB):")
print(f"  Mean  L={mean_lab[0]:.1f}  a={mean_lab[1]:.1f}  b={mean_lab[2]:.1f}")
print(f"  Std   L={std_lab[0]:.1f}   a={std_lab[1]:.1f}   b={std_lab[2]:.1f}")
print(f"  σ gate width: {SIGMA_THRESH} (pixels > {SIGMA_THRESH}σ are excluded as background)")

# Visualise LAB channels
L_ch = img_lab_float[:, :, 0]
a_ch = img_lab_float[:, :, 1]
b_ch = img_lab_float[:, :, 2]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("Stage 2 · LAB Colour Space — seed pixels used to learn the leaf model",
             fontsize=13, fontweight="bold")
axes[0].imshow(_bgr2rgb(img_resized));  axes[0].set_title("Input"); axes[0].axis("off")
axes[1].imshow(L_ch, cmap="gray");     axes[1].set_title("L channel (lightness)"); axes[1].axis("off")
axes[2].imshow(a_ch, cmap="RdYlGn");  axes[2].set_title("a channel (green←→red)"); axes[2].axis("off")
axes[3].imshow(b_ch, cmap="YlOrBr");  axes[3].set_title("b channel (blue←→yellow)"); axes[3].axis("off")

# Mark seed mean on L channel
for ax, val, mn, std in zip([axes[1], axes[2], axes[3]],
                             ["L", "a", "b"], mean_lab, std_lab):
    ax.set_xlabel(f"Leaf mean={mn:.1f}  std={std:.1f}", fontsize=9)
plt.tight_layout(); plt.show()


### 3.3 · Stage 3 — Candidate Pixel Map
Two gates combined:
- **Colour-model gate**: pixel must be within σ_thresh standard deviations of the learned leaf LAB model.
- **Saturation gate**: S > 25 (excludes achromatic shadows and white paper).
- **Paper-gap gate**: rejects pixels that are simultaneously pale (L > 160) AND achromatic (S < 30).


In [ ]:
candidate = _build_candidate_map(img_lab_float, hsv, mean_lab, std_lab, SIGMA_THRESH)
candidate[is_padding] = 0

# Build individual gate maps for inspection
diff    = np.abs(img_lab_float - mean_lab)
z       = diff / std_lab
sigma_gate_vis = ((z.max(axis=2) < SIGMA_THRESH) * 255).astype(np.uint8)

s_ch  = hsv[:, :, 1].astype(np.float32)
L_ch2 = img_lab_float[:, :, 0]
sat_gate_vis     = ((s_ch > 25) * 255).astype(np.uint8)
paper_gap_vis    = (((s_ch < 30) & (L_ch2 > 160)) * 255).astype(np.uint8)

cand_pct = (candidate > 0).sum() / img_area * 100

print(f"Candidate pixels: {(candidate > 0).sum()}  ({cand_pct:.1f}% of image)")

_show(
    [sigma_gate_vis, sat_gate_vis, paper_gap_vis, candidate,
     _overlay_mask(img_resized, candidate, colour=(255, 165, 0))],
    ["σ-model gate\n(within 2.5σ of leaf LAB)",
     "Saturation gate\n(S > 25 → real colour)",
     "Paper-gap rejection\n(pale AND achromatic)",
     "Stage 3 · Candidate map\n(all gates combined)",
     "Candidate overlay"],
    cmap_list=["gray", "gray", "gray", "gray", None],
    figsize=(25, 5),
    suptitle="Stage 3 · Candidate Pixel Map"
)


### 3.4 · Stage 4 — Region Growing
Iteratively dilates the seed mask, constrained to candidate pixels.
Stops early when no new pixels are added (convergence).


In [ ]:
grown = _grow_seed(seed, candidate, n_iterations=40, kernel_size=5)
grown = _remove_noise(grown, min_frac=MIN_COMP_FRAC, img_area=img_area)

grown_pct = (grown > 0).sum() / img_area * 100
print(f"Grown pixels : {(grown > 0).sum()}  ({grown_pct:.1f}% of image)")
print(f"Growth from seed: {seed_cov:.1f}% → {grown_pct:.1f}%  (+{grown_pct - seed_cov:.1f}%)")

_show(
    [seed, candidate, grown, _overlay_mask(img_resized, grown, colour=(0, 200, 255))],
    ["Seed (start)", "Candidate map\n(growth boundary)",
     "Stage 4 · Grown mask", "Grown overlay"],
    cmap_list=["gray", "gray", "gray", None],
    figsize=(20, 5),
    suptitle="Stage 4 · Region Growing (seed expands within candidate boundary)"
)


### 3.5 · Stage 5 — Tight vs Loose Structure Selection
Chooses between:
- **Tight close** (3×3 kernel): preserves inter-leaflet gaps in pinnate species.
- **Loose close** (5×5 kernel): merges nearby regions for trifoliate/sparse species.

Selection rule: use tight if ≥ 3 components AND tight area ≥ 25 % of loose area.


In [ ]:
leaflet_mask, mask_choice, n_tight, tight_area, loose_area = _select_structure(
    grown, MIN_COMP_FRAC, img_area
)

# Build both for comparison
k3_s = np.ones((3, 3), np.uint8)
k5_s = np.ones((5, 5), np.uint8)
m_tight = _remove_noise(
    cv2.morphologyEx(grown, cv2.MORPH_CLOSE, k3_s, iterations=1),
    min_frac=MIN_COMP_FRAC, img_area=img_area
)
m_loose = _remove_noise(
    cv2.morphologyEx(grown, cv2.MORPH_CLOSE, k5_s, iterations=1),
    min_frac=MIN_COMP_FRAC, img_area=img_area
)

print(f"Mask choice   : '{mask_choice}'  ({'tight=3×3' if mask_choice=='tight' else 'loose=5×5'})")
print(f"Tight comps   : {n_tight}   area={tight_area} px")
print(f"Loose area    : {loose_area} px")
print(f"Area ratio    : {tight_area/max(loose_area,1)*100:.1f}%  (threshold: 25%)")

_show(
    [m_tight, m_loose, leaflet_mask,
     _overlay_mask(img_resized, leaflet_mask, colour=(0, 255, 128))],
    [f"Tight close (3×3)\n{n_tight} components",
     "Loose close (5×5)",
     f"Stage 5 · Selected: '{mask_choice}'",
     "Leaflet mask overlay"],
    cmap_list=["gray", "gray", "gray", None],
    figsize=(20, 5),
    suptitle="Stage 5 · Tight vs Loose Structure Selection"
)


### 3.6 · Stage 6 — Rachis / Petiole Detection
Detects the woody central stem separately using:
- **Tier A (brown/tan)**: LAB b > 133 (yellow/brown shift) + S > 35 + 50 < L < 150.
- **Tier B (green stem)**: ExG in (3, 18) + S > 20 + L < 140.
- **Proximity gate**: only rachis pixels within 15 px of the leaflet mask are kept.


In [ ]:
rachis_mask = _build_rachis_mask(
    img_resized, img_lab_float, hsv, leaflet_mask, is_padding, proximity_px=15
)

# Build individual tier masks for inspection
img_f2 = img_resized.astype(np.float32)
exg2   = 2.0 * img_f2[:,:,1] - img_f2[:,:,2] - img_f2[:,:,0]
b_ch2  = img_lab_float[:, :, 2]
s_ch2  = hsv[:, :, 1].astype(np.float32)
L_ch3  = img_lab_float[:, :, 0]

tier_a = ((b_ch2 > 133) & (s_ch2 > 35) & (L_ch3 > 50) & (L_ch3 < 150)).astype(np.uint8) * 255
tier_b = ((exg2 > 3) & (exg2 < 18) & (s_ch2 > 20) & (L_ch3 < 140)).astype(np.uint8) * 255
k_prox = np.ones((31, 31), np.uint8)
proximity_zone = cv2.dilate(leaflet_mask, k_prox, iterations=1)

rachis_px = int((rachis_mask > 0).sum())
print(f"Rachis pixels : {rachis_px}  ({rachis_px / img_area * 100:.3f}% of image)")

# Overlay rachis on leaflet
combined_vis = _bgr2rgb(img_resized).copy()
combined_vis[leaflet_mask > 0] = (
    combined_vis[leaflet_mask > 0] * 0.6 + np.array([0, 180, 0]) * 0.4
).astype(np.uint8)
combined_vis[rachis_mask > 0] = (
    combined_vis[rachis_mask > 0] * 0.5 + np.array([200, 100, 0]) * 0.5
).astype(np.uint8)

_show(
    [tier_a, tier_b, proximity_zone, rachis_mask],
    ["Tier A candidate\n(brown/tan: b>133, S>35)",
     "Tier B candidate\n(green stem: ExG 3–18)",
     "Proximity zone\n(15 px around leaflets)",
     "Stage 6 · Rachis mask"],
    cmap_list=["gray", "gray", "gray", "gray"],
    figsize=(20, 5),
    suptitle="Stage 6 · Rachis / Petiole Detection"
)
fig, ax = plt.subplots(1, 1, figsize=(6, 6))
ax.imshow(combined_vis)
ax.set_title("Leaflet (green) + Rachis (orange) overlay", fontweight="bold")
ax.axis("off"); plt.show()


### 3.7 · Stage 7 — Union: Leaflet ∪ Rachis
Combines the leaflet and rachis masks with a bitwise OR.


In [ ]:
combined_mask = cv2.bitwise_or(leaflet_mask, rachis_mask)

print(f"Leaflet pixels : {(leaflet_mask > 0).sum()}")
print(f"Rachis pixels  : {(rachis_mask > 0).sum()}")
print(f"Union pixels   : {(combined_mask > 0).sum()}  (leaflet + rachis, any overlap counted once)")

_show(
    [leaflet_mask, rachis_mask, combined_mask,
     _overlay_mask(img_resized, combined_mask, colour=(100, 255, 100))],
    ["Leaflet mask", "Rachis mask",
     "Stage 7 · Union mask", "Union overlay"],
    cmap_list=["gray", "gray", "gray", None],
    figsize=(20, 5),
    suptitle="Stage 7 · Leaflet ∪ Rachis Union"
)


### 3.8 · Stage 8 — Hole Fill
Border-seeded flood fill on the inverted mask closes enclosed holes
(light bleed, translucent spots) WITHOUT filling inter-leaflet gaps
(those gaps touch the image border and are not enclosed).

**Order matters**: hole fill runs BEFORE `_remove_noise` so filled interiors
are not deleted as small isolated components.


In [ ]:
filled_mask = _fill_holes(combined_mask)

holes_filled = int((filled_mask > 0).sum()) - int((combined_mask > 0).sum())
print(f"Pixels added by hole fill: {holes_filled}")

# Show difference
hole_pixels = ((filled_mask > 0) & ~(combined_mask > 0)).astype(np.uint8) * 255

_show(
    [combined_mask, hole_pixels, filled_mask,
     _overlay_mask(img_resized, filled_mask, colour=(180, 255, 180))],
    ["Before hole fill", "Filled holes only\n(new pixels in white)",
     "Stage 8 · After hole fill", "Hole-filled overlay"],
    cmap_list=["gray", "gray", "gray", None],
    figsize=(20, 5),
    suptitle="Stage 8 · Enclosed Hole Fill (border flood-fill)"
)


### 3.9 · Stage 9 — Final Clean + Paper-Leak Guard
- Light morphological close (3×3, 1 iteration).
- Noise removal: drops components < 0.1 % of image area (< 262 px at 512²).
- Paper-leak veto: any pixel with L > 175 AND S < 25 is forced to background (cannot be biological tissue).
- Padding exclusion: border pixels from letterboxing are cleared.


In [ ]:
# Replicate Stage 9 exactly
mask_close  = cv2.morphologyEx(filled_mask, cv2.MORPH_CLOSE, k3, iterations=1)
mask_clean  = _remove_noise(mask_close, min_frac=MIN_COMP_FRAC, img_area=img_area)
is_paper_leak = (img_lab_float[:,:,0] > 175) & (hsv[:,:,1].astype(np.float32) < 25)
mask_clean[is_paper_leak] = 0
mask_final_rebuilt = _remove_noise(mask_clean, min_frac=MIN_COMP_FRAC, img_area=img_area)
mask_final_rebuilt[is_padding] = 0

coverage_pct = (mask_final_rebuilt > 0).sum() / img_area * 100
print(f"Final coverage : {coverage_pct:.2f}% of image area")

# Use the official API call (should match)
mask_final, mask_choice_official, mask_diag = select_mask(img_resized)
qc_passed, qc_reason = qc_check(mask_diag)
print(f"QC passed      : {qc_passed}  {qc_reason if not qc_passed else ''}")
print(f"Mask choice    : {mask_choice_official}")

_show(
    [filled_mask, mask_final,
     _overlay_mask(img_resized, mask_final, colour=(0, 255, 0))],
    ["Before final clean", "Stage 9 · Final mask",
     "Final mask overlay"],
    cmap_list=["gray", "gray", None],
    figsize=(15, 5),
    suptitle=f"Stage 9 · Final Mask  (coverage={coverage_pct:.1f}%,  choice='{mask_choice_official}')"
)


### 3.10 · Masking Diagnostics Summary


In [ ]:
print("=" * 55)
print("MASKING DIAGNOSTICS")
print("=" * 55)
for k, v in mask_diag.items():
    print(f"  {k:<26s}: {v}")
print("=" * 55)

# All 9 stages side-by-side summary
stage_masks = [
    img_resized,
    seed,
    candidate,
    grown,
    leaflet_mask,
    rachis_mask,
    combined_mask,
    filled_mask,
    mask_final,
]
stage_labels = [
    "0. Input", "1. Seed", "2→3. Candidate",
    "4. Grown", "5. Leaflet\n(tight/loose)",
    "6. Rachis", "7. Union",
    "8. Hole fill", "9. Final mask",
]
cmaps = [None, "gray", "gray", "gray", "gray", "gray", "gray", "gray", "gray"]

fig, axes = plt.subplots(1, 9, figsize=(45, 5))
fig.suptitle("All 9 Masking Stages at a Glance", fontsize=14, fontweight="bold")
for ax, img, title, cm in zip(axes, stage_masks, stage_labels, cmaps):
    if img.ndim == 3:
        ax.imshow(_bgr2rgb(img))
    else:
        ax.imshow(img, cmap=cm)
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.axis("off")
plt.tight_layout(); plt.show()


## 4 · Preprocessing — Step 3: Species-ID Enhancement
Three stages applied to the masked image ONLY (not to the colour-feature input):
1. **Bilateral filter** — smooths within-leaflet noise, preserves vein edges.
2. **CLAHE on L channel** — local contrast boost without hue shift.
3. **Unsharp mask** — sharpens vein boundaries for skeleton extraction.


In [ ]:
from preprocessing.species_id.enhance import enhance_for_species_id
from preprocessing.config import BILATERAL_D, BILATERAL_SIGMA, CLAHE_CLIP, CLAHE_TILE, UNSHARP_SIGMA, UNSHARP_STRENGTH

img_masked = cv2.bitwise_and(img_resized, img_resized, mask=mask_final)

# Stage-by-stage
img_bilateral = cv2.bilateralFilter(img_masked, d=BILATERAL_D,
                                     sigmaColor=BILATERAL_SIGMA,
                                     sigmaSpace=BILATERAL_SIGMA)

lab_e = cv2.cvtColor(img_bilateral, cv2.COLOR_BGR2LAB)
l, a_e, b_e = cv2.split(lab_e)
clahe_obj = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=CLAHE_TILE)
l_clahe = clahe_obj.apply(l)
img_clahe = cv2.cvtColor(cv2.merge([l_clahe, a_e, b_e]), cv2.COLOR_LAB2BGR)

blur = cv2.GaussianBlur(img_clahe, (0, 0), sigmaX=UNSHARP_SIGMA)
img_sharp = cv2.addWeighted(img_clahe, UNSHARP_STRENGTH, blur, -(UNSHARP_STRENGTH - 1.0), 0)
img_sharp = cv2.bitwise_and(img_sharp, img_sharp, mask=mask_final)

_show(
    [img_masked, img_bilateral, img_clahe, img_sharp],
    ["Masked raw input",
     f"After bilateral filter\n(d={BILATERAL_D}, σ={BILATERAL_SIGMA})",
     f"After CLAHE on L\n(clip={CLAHE_CLIP}, tile={CLAHE_TILE})",
     f"After unsharp mask\n(σ={UNSHARP_SIGMA}, strength={UNSHARP_STRENGTH})"],
    figsize=(20, 5),
    suptitle="Step 3 · Species-ID Enhancement Pipeline"
)


## 5 · Feature Extraction

### 5.1 · Shape Features
Dimensionless geometric descriptors from the binary mask contour:
aspect ratio, circularity, solidity, convexity (perimeter ratio), compactness, elongation, and 7 Hu moments.


In [ ]:
from feature_extraction.species_id.shape import extract_shape_features

shape_f = extract_shape_features(mask_final)

# ── Visualise key shape geometry ──────────────────────────────────────────
cnts, _ = cv2.findContours(mask_final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
cnt = max(cnts, key=cv2.contourArea)
x, y, w, h = cv2.boundingRect(cnt)
hull = cv2.convexHull(cnt)

vis_shape = _bgr2rgb(img_resized).copy()
cv2.drawContours(vis_shape, [cnt],  -1, (255, 50,  50), 2)   # contour — red
cv2.drawContours(vis_shape, [hull], -1, (50,  50, 255), 2)   # hull — blue
cv2.rectangle(vis_shape, (x, y), (x+w, y+h), (255, 220, 0), 2)  # bbox — yellow
if len(cnt) >= 5:
    ell = cv2.fitEllipse(cnt)
    cv2.ellipse(vis_shape, ell, (0, 220, 0), 2)              # ellipse — green

# Print features
print("Shape features:")
for k, v in shape_f.items():
    print(f"  {k:<18s}: {v:.6f}")

patches = [
    mpatches.Patch(color=(1,.2,.2), label="Contour"),
    mpatches.Patch(color=(.2,.2,1), label="Convex hull"),
    mpatches.Patch(color=(1,.86,0), label="Bounding box"),
    mpatches.Patch(color=(0,.86,0), label="Fitted ellipse"),
]
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("5.1 · Shape Features", fontsize=13, fontweight="bold")
axes[0].imshow(vis_shape); axes[0].set_title("Geometry overlay"); axes[0].legend(handles=patches, fontsize=8)
axes[0].axis("off")
# Bar chart — key ratios
keys_plot = ["aspect_ratio","circularity","solidity","convexity","compactness","elongation"]
vals_plot = [shape_f[k] for k in keys_plot]
axes[1].barh(keys_plot, vals_plot, color="steelblue")
axes[1].set_xlim(0, max(max(vals_plot)*1.15, 1.1))
axes[1].set_title("Shape ratio features"); axes[1].axvline(1.0, color="red", linestyle="--", alpha=0.5)
axes[1].invert_yaxis()
# Hu moments
hu_keys = [k for k in shape_f if k.startswith("hu_")]
hu_vals = [shape_f[k] for k in hu_keys]
axes[2].bar(hu_keys, hu_vals, color="darkorange")
axes[2].set_title("Log-normalised Hu moments"); axes[2].tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.show()


### 5.2 · Colour Features
Extracted from the **raw letterboxed image** (NOT the enhanced image) to preserve colour integrity.
Uses median + IQR (shadow-robust) statistics across BGR, HSV, LAB channels, ExG index, and a 6-bin hue histogram.


In [ ]:
from feature_extraction.species_id.colour import extract_colour_features

colour_f = extract_colour_features(img_resized, mask_final)   # RAW image

# Print features
print(f"Colour feature count: {len(colour_f)}")
print("\nKey statistics (median):")
for k in sorted(colour_f):
    if "median" in k or "dominant" in k or "hue_hist" in k:
        print(f"  {k:<30s}: {colour_f[k]:.4f}")

# ── Visualise ─────────────────────────────────────────────────────────────
img_hsv_vis = cv2.cvtColor(img_resized, cv2.COLOR_BGR2HSV)
img_lab_vis = cv2.cvtColor(img_resized, cv2.COLOR_BGR2LAB)
px = mask_final > 0

fig = plt.figure(figsize=(22, 8))
fig.suptitle("5.2 · Colour Features", fontsize=13, fontweight="bold")
gs = GridSpec(2, 5, figure=fig)

# Colour space panels
for ci, (ch_img, title) in enumerate(zip(
    [img_resized[:,:,1], img_hsv_vis[:,:,0], img_hsv_vis[:,:,1],
     img_hsv_vis[:,:,2], img_lab_vis[:,:,0]],
    ["G channel (BGR)", "H channel (HSV)", "S channel (HSV)",
     "V channel (HSV)", "L channel (LAB)"]
)):
    ax = fig.add_subplot(gs[0, ci])
    ax.imshow(ch_img, cmap="gray"); ax.set_title(title, fontsize=9); ax.axis("off")

# Hue histogram
hue_vals = img_hsv_vis[:,:,0][px].astype(np.float32)
ax_hue = fig.add_subplot(gs[1, :2])
hist6 = np.array([colour_f[f"hue_hist_{bi:02d}"] for bi in range(6)])
bin_labels = ["0–30°\nred-orange","30–60°\nyellow-grn","60–90°\ngreen",
              "90–120°\nblue-grn","120–150°\nblue","150–180°\nmagenta"]
ax_hue.bar(range(6), hist6, color=["tomato","yellowgreen","limegreen","teal","steelblue","orchid"])
ax_hue.set_xticks(range(6)); ax_hue.set_xticklabels(bin_labels, fontsize=8)
ax_hue.set_title(f"6-bin Hue Histogram  (dominant_hue={colour_f['dominant_hue']:.0f}°, peak_frac={colour_f['hue_peak_fraction']:.2f})")
ax_hue.set_ylabel("Normalised frequency")

# IQR stats comparison
ax_iqr = fig.add_subplot(gs[1, 2:])
channels = ["bgr_b","bgr_g","bgr_r","hsv_h","hsv_s","hsv_v","lab_l"]
medians  = [colour_f[f"{c}_median"] for c in channels]
iqrs     = [colour_f[f"{c}_iqr"]    for c in channels]
x_pos    = range(len(channels))
ax_iqr.bar(x_pos, medians, yerr=iqrs, capsize=5, color="cornflowerblue", alpha=0.8)
ax_iqr.set_xticks(x_pos); ax_iqr.set_xticklabels(channels, rotation=30, ha="right", fontsize=9)
ax_iqr.set_title("Median ± IQR per channel (foreground pixels)")
ax_iqr.set_ylabel("Pixel value")
plt.tight_layout(); plt.show()


### 5.3 · Texture Features
Uses the **enhanced image**. Two descriptors:
- **GLCM** (Grey-Level Co-occurrence Matrix): contrast, homogeneity, energy, correlation — shadow pixels median-filled before computation.
- **LBP** (Local Binary Pattern): rotation-invariant, illumination-invariant histogram of local texture codes.


In [ ]:
from feature_extraction.species_id.texture import extract_texture_features
from skimage.feature import local_binary_pattern
from preprocessing.config import LBP_RADIUS, LBP_POINTS

texture_f = extract_texture_features(img_sharp, mask_final)

print(f"Texture feature count: {len(texture_f)}")
print("\nGLCM features:")
for k in sorted(texture_f):
    if "glcm" in k:
        print(f"  {k:<30s}: {texture_f[k]:.6f}")

# ── Visualise ─────────────────────────────────────────────────────────────
gray_sharp = cv2.cvtColor(img_sharp, cv2.COLOR_BGR2GRAY)
px_mask    = mask_final > 0
lbp        = local_binary_pattern(gray_sharp, LBP_POINTS, LBP_RADIUS, method="uniform")
lbp_vis    = (lbp / lbp.max() * 255).astype(np.uint8)
lbp_vis[~px_mask] = 0

# Confident mask (mirrors texture.py shadow exclusion)
hsv_v_tex = cv2.cvtColor(img_sharp, cv2.COLOR_BGR2HSV)[:,:,2]
shadow_tex = (hsv_v_tex < 40) & px_mask
conf_mask  = px_mask & ~shadow_tex

fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle("5.3 · Texture Features", fontsize=13, fontweight="bold")
axes[0].imshow(_bgr2rgb(img_sharp));            axes[0].set_title("Enhanced input"); axes[0].axis("off")
axes[1].imshow(gray_sharp, cmap="gray");        axes[1].set_title("Greyscale"); axes[1].axis("off")
axes[2].imshow(shadow_tex, cmap="hot");         axes[2].set_title("Shadow pixels\n(V < 40, median-filled)"); axes[2].axis("off")
axes[3].imshow(lbp_vis, cmap="viridis");        axes[3].set_title(f"LBP map\n(r={LBP_RADIUS}, pts={LBP_POINTS})"); axes[3].axis("off")

# LBP histogram
n_bins = LBP_POINTS + 2
lbp_vals = lbp[px_mask]
lbp_hist, _ = np.histogram(lbp_vals, bins=n_bins, range=(0, n_bins))
lbp_hist = lbp_hist / (lbp_hist.sum() + 1e-6)
axes[4].bar(range(n_bins), lbp_hist, color="mediumpurple")
axes[4].set_title("LBP histogram\n(uniform patterns)"); axes[4].set_xlabel("LBP code")
plt.tight_layout(); plt.show()


### 5.4 · Vein Features
ROI-upscale pipeline:
1. Crop tightly to leaf bounding box → upscale to 512 px (INTER_CUBIC).
2. CLAHE → black top-hat (15 px ellipse) → adaptive threshold → skeletonise.
3. Density ratios normalised by leaf area (not image area) for scale-invariance.


In [ ]:
from feature_extraction.species_id.vein import (
    extract_vein_features, _get_padded_bbox, _upscale_to_work_size, _build_vein_map, WORK_SIZE
)

vein_f, vein_skel, vein_binary = extract_vein_features(img_sharp, mask_final)

print(f"Vein feature count: {len(vein_f)}")
print("\nVein features:")
for k, v in vein_f.items():
    print(f"  {k:<30s}: {v:.6f}")

# ── Step-by-step vein internals ───────────────────────────────────────────
bbox = _get_padded_bbox(mask_final)
x1, y1, x2, y2 = bbox
gray_sharp_full = cv2.cvtColor(img_sharp, cv2.COLOR_BGR2GRAY)
crop_gray = gray_sharp_full[y1:y2, x1:x2]
crop_mask = mask_final[y1:y2, x1:x2]
crop_rgb  = _bgr2rgb(img_sharp)[y1:y2, x1:x2]

gray_work, scale = _upscale_to_work_size(crop_gray)
mask_work = cv2.resize(crop_mask, (gray_work.shape[1], gray_work.shape[0]),
                       interpolation=cv2.INTER_NEAREST)

# Intermediate steps
clahe_vein = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
gray_eq    = clahe_vein.apply(gray_work)
k_bthat    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
tophat     = cv2.morphologyEx(gray_eq, cv2.MORPH_BLACKHAT, k_bthat)
tophat_masked = cv2.bitwise_and(tophat, tophat, mask=mask_work)
blk = max(11, (gray_work.shape[0] // 20) | 1)
vein_bin_work = cv2.adaptiveThreshold(tophat_masked, 255,
                                       cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                       cv2.THRESH_BINARY, blk, -2)
vein_bin_work = cv2.bitwise_and(vein_bin_work, vein_bin_work, mask=mask_work)
vein_sk_work  = skeletonize(vein_bin_work > 0).astype(np.uint8) * 255

# Overlay skeleton on leaf crop
sk_full_rgb = _bgr2rgb(img_resized).copy()
if vein_skel is not None:
    sk_full_rgb[vein_skel > 0] = [255, 50, 50]

_show(
    [crop_rgb, gray_eq, tophat_masked, vein_bin_work, vein_sk_work],
    ["Leaf crop (enhanced)", f"CLAHE equalised\n(scale={scale:.2f}×)",
     "Black top-hat\n(vein enhancement)", "Adaptive threshold\n(vein binary)",
     "Skeleton\n(1-px vein centrelines)"],
    cmap_list=[None, "gray", "gray", "gray", "gray"],
    figsize=(25, 5),
    suptitle="5.4 · Vein Extraction — Internal Steps (on upscaled crop)"
)
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 6))
axes2[0].imshow(sk_full_rgb); axes2[0].set_title("Vein skeleton overlay (red) on original"); axes2[0].axis("off")
axes2[1].bar(["vein_density","vein_length_ratio","vein_branch_density","vein_end_point_density"],
             [vein_f.get(k, 0) for k in ["vein_density","vein_length_ratio","vein_branch_density","vein_end_point_density"]],
             color="firebrick")
axes2[1].set_title("Vein density features"); axes2[1].tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()


### 5.5 · Whole-Leaf Structural Features
Leaflet arrangement descriptors based on connected components of the mask:
aspect, area CV, area max/min ratio, LR symmetry, spacing CV, and normalised component count.


In [ ]:
from feature_extraction.species_id.whole_leaf import extract_whole_leaf_features
from preprocessing.config import MIN_COMP_FRAC

whole_f = extract_whole_leaf_features(mask_final)

print(f"Whole-leaf feature count: {len(whole_f)}")
print("\nWhole-leaf features:")
for k, v in whole_f.items():
    print(f"  {k:<30s}: {v:.6f}")

# ── Visualise connected components ────────────────────────────────────────
img_area_wl = mask_final.shape[0] * mask_final.shape[1]
min_px_wl   = int(img_area_wl * MIN_COMP_FRAC)
n_comp, labels, stats, centroids = cv2.connectedComponentsWithStats(mask_final)

comp_vis = _bgr2rgb(img_resized).copy()
colours  = plt.cm.tab10.colors
sig_comps = []
for i in range(1, n_comp):
    if stats[i, cv2.CC_STAT_AREA] >= min_px_wl:
        sig_comps.append(i)
        col = tuple(int(c * 255) for c in colours[(i-1) % 10])
        x_, y_, w_, h_, _ = stats[i]
        cv2.rectangle(comp_vis, (x_, y_), (x_+w_, y_+h_), col, 2)
        cx, cy = int(centroids[i][0]), int(centroids[i][1])
        cv2.circle(comp_vis, (cx, cy), 5, col, -1)
        cv2.putText(comp_vis, str(i), (cx+6, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.5, col, 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("5.5 · Whole-Leaf Structural Features", fontsize=13, fontweight="bold")
axes[0].imshow(comp_vis); axes[0].set_title(f"Connected components ({len(sig_comps)} significant)"); axes[0].axis("off")

# Area distribution
if sig_comps:
    areas = [stats[i, cv2.CC_STAT_AREA] for i in sig_comps]
    axes[1].bar([f"C{i}" for i in sig_comps], areas, color=[colours[(j) % 10] for j in range(len(sig_comps))])
    axes[1].set_title("Component areas (px)"); axes[1].set_ylabel("Area (px)")
    axes[1].tick_params(axis="x", rotation=45)

# Feature summary
feat_keys = list(whole_f.keys())
feat_vals = list(whole_f.values())
axes[2].barh(feat_keys, feat_vals, color="teal")
axes[2].set_title("Whole-leaf features"); axes[2].invert_yaxis()
axes[2].axvline(0, color="black", linewidth=0.5)
plt.tight_layout(); plt.show()


## 6 · Full Pipeline Run & Feature Summary

In [ ]:
from preprocessing.species_id.pipeline import run_pipeline

feats, info = run_pipeline(IMAGE_PATH, SPECIES, VIEW_SIDE)

if feats is None:
    print(f"[FAIL] QC did not pass: {info['qc_reason']}")
else:
    meta_keys = {"species", "view_side", "image_path", "mask_choice",
                 "coverage_pct", "vein_coverage_pct", "vein_roi_scale"}
    feat_keys = [k for k in feats if k not in meta_keys]
    print(f"✓ Pipeline complete")
    print(f"  Total features  : {len(feat_keys)}")
    print(f"  Mask choice     : {info['mask_choice']}")
    print(f"  Coverage        : {info['mask_diag']['coverage_pct']:.1f}%")
    print(f"  QC passed       : {info['qc_passed']}")
    print()

    # Group by prefix
    groups = {}
    for k in feat_keys:
        prefix = k.split("_")[0]
        groups.setdefault(prefix, []).append(k)

    print("Feature counts by group:")
    for g, ks in groups.items():
        print(f"  {g:<10s}: {len(ks)} features")

    # Final composite figure
    fig, axes = plt.subplots(1, 4, figsize=(24, 6))
    fig.suptitle("Pipeline Summary — Single Leaf", fontsize=14, fontweight="bold")
    axes[0].imshow(_bgr2rgb(info["img_orig"])); axes[0].set_title("Original input"); axes[0].axis("off")
    axes[1].imshow(_bgr2rgb(info["img_resized"])); axes[1].set_title("Letterboxed 512×512"); axes[1].axis("off")
    axes[2].imshow(info["mask_final"], cmap="gray"); axes[2].set_title(f"Final mask\n({info['mask_choice']}, {info['mask_diag']['coverage_pct']:.1f}%)"); axes[2].axis("off")
    masked_rgb = _bgr2rgb(info["img_masked"])
    if info["img_sharp"] is not None:
        masked_rgb = _bgr2rgb(info["img_sharp"])
    axes[3].imshow(masked_rgb); axes[3].set_title("Enhanced (species-ID input)"); axes[3].axis("off")
    plt.tight_layout(); plt.show()


## 7 · Full Feature Table

In [ ]:
import pandas as pd

if feats is not None:
    meta_keys = {"species", "view_side", "image_path", "mask_choice",
                 "coverage_pct", "vein_coverage_pct", "vein_roi_scale"}
    feat_only = {k: v for k, v in feats.items() if k not in meta_keys}

    df = pd.DataFrame([feat_only]).T.reset_index()
    df.columns = ["feature", "value"]
    df["group"] = df["feature"].str.split("_").str[0]
    df = df[["group","feature","value"]].sort_values(["group","feature"])

    pd.set_option("display.max_rows", 200)
    pd.set_option("display.float_format", "{:.6f}".format)
    display(df)
    print(f"\nTotal: {len(df)} features")
